# Aquaculture Model Training

This notebook demonstrates the training workflow for the aquaculture ML framework using actual competition data.


## 1. Setup and Configuration


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import os
import random
from pathlib import Path
import sys

# Add the parent directory to the system path to import local modules
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# Import our custom modules
from aquaculture.feature_engineering import AquacultureFeatureEngineer
from src.config import TrainingConfig
from src.trainer import Trainer

# For reproducibility
np.random.seed(42)
random.seed(42)

# Define where your data files live.
# Adjust this path if your data are located elsewhere.
DATA_DIR = Path("../data")      # relative to the notebook's working directory
# Verify the directory exists
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Data directory not found: {DATA_DIR.resolve()}")

# Define the path to the experiment directory where models and results will be saved
EXPERIMENT_DIR = Path("../experiments")  # relative to the notebook's working directory
# Create the experiment directory if it doesn't exist
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Data Loading and Preparation


In [ ]:
# Load training data from CSV file
print("Loading training data...")
train_df = pd.read_csv(DATA_DIR / 'Train.csv')
print(f"Training data shape: {train_df.shape}")
print(f"Training data columns: {list(train_df.columns)}")

# Load test data from CSV file
print("Loading test data...")
test_df = pd.read_csv(DATA_DIR / 'Test.csv')
print(f"Test data shape: {test_df.shape}")

# Check submission format
sample_submission = pd.read_csv(DATA_DIR / 'SampleSubmission.csv')
print(f"Sample submission shape: {sample_submission.shape}")
print(f"Sample submission columns: {list(sample_submission.columns)}")
print("\nSample submission head:")
print(sample_submission.head())

# Prepare data for training
print("Preparing data for training...")
# The target column is 'label' in the training data
# Feature columns are all columns except ID and label
feature_cols = [col for col in train_df.columns if col not in ['ID', 'label']]
X = train_df[feature_cols].values

# Get target variable - binary classification: 0 (no pond) or 1 (pond)
print("Extracting target variable from 'label' column...")
y = train_df['label'].values
print(f"Target variable shape: {y.shape}")
print(f"Target distribution: {np.bincount(y.astype(int)) if len(y) > 0 else 'empty'}")


## 3. Model Training and Evaluation


In [ ]:
# Create a configuration object
config = TrainingConfig()                     # <-- instantiate the config

# Set experiment directory in config
config.experiment_dir = EXPERIMENT_DIR

# Initialize trainer
trainer = Trainer(config)

# Train models
print("Training models...")
trainer.fit(X, y)

# Check if training was successful
if trainer.model is not None:
    print("\n✓ Model training successful!")
    print(f"✓ Best model type: {config.model_type}")
    print(f"✓ Number of features: {len(trainer.feature_names) if trainer.feature_names else 'Unknown'}")
    
    # Show best hyperparameters
    if trainer.best_params:
        print("\nBest hyperparameters:")
        for param, value in trainer.best_params.items():
            print(f"  {param}: {value}")
    
    # Show experiment directory
    print(f"\nExperiment saved to: {trainer.experiment_dir}")
else:
    print("\n✗ Model training failed!")

In [ ]:
# Evaluate training performance with observation simulation (matches training conditions)
print("\n=== Training set evaluation (with observation stimulation) ===")
train_preds = trainer.predict(X, training=True)
train_probas = trainer.predict_proba(X, training=True)[:, 1]

from src.metrics import calculate_metrics, competition_score
metrics = calculate_metrics(y, train_probas)
print("Metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

comp = competition_score(y, train_probas)
print(f"  competition_score: {comp:.4f}")
